In [1]:
from PIL import Image
from rl_benchmarks.models import iBOTViT
from openslide import open_slide
from openslide.deepzoom import DeepZoomGenerator
import pathlib
from tqdm import tqdm
from PIL import Image
import numpy as np
Image.MAX_IMAGE_PIXELS = None
from torchvision import transforms
import torch
from torch.utils.data import Dataset
import os
from multiprocessing import Pool
import umap
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from rl_benchmarks.utils.linear_evaluation import get_binary_class_metrics, get_bootstrapped_metrics

from PIL import Image
import pathlib
from tqdm import tqdm
from PIL import Image
import numpy as np
Image.MAX_IMAGE_PIXELS = None
from torchvision import transforms
import torch
from torch.utils.data import Dataset
import os
from multiprocessing import Pool
# import umap
import numpy as np
import matplotlib.pyplot as plt
import sklearn

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms

from sklearn.metrics import f1_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix, cohen_kappa_score, accuracy_score
import torch.nn.functional as F
import sys
import time

import shutil
import os

# import albumentations as A
# from albumentations.pytorch import ToTensorV2
import cv2
import time
from sklearn.preprocessing import label_binarize

import timm
from metrics import report
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Subset, Dataset
from tqdm import tqdm
from rl_benchmarks.metrics import *
from sklearn.metrics import balanced_accuracy_score, cohen_kappa_score, f1_score

from rl_benchmarks.trainers.torch_trainer import TorchTrainer
from rl_benchmarks.models.slide_models.meanpool import MeanPool
from rl_benchmarks.models.slide_models.chowder import Chowder
from rl_benchmarks.models.slide_models.dsmil import DSMIL
from rl_benchmarks.models.slide_models.abmil import ABMIL
from rl_benchmarks.models.slide_models.hiptmil import HIPTMIL
from rl_benchmarks.models.slide_models.transmil import TransMIL

from pathlib import Path
from metrics import report

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
    TaskType,
)

import sys
sys.path.append('/home/yuxin/bme/BCaCAD/model')
from patch_based_test.img import QiLuROI

def get_model(model_name, in_dim, out_dim):
    if model_name == 'abmil':
        model = ABMIL(in_dim, out_dim, d_model_attention=128,temperature= 1.0, mlp_hidden= [128, 64])
    elif model_name == 'chowder':
        model = Chowder(in_dim, out_dim, n_top= 2,n_bottom= 2,tiles_mlp_hidden= [128], mlp_hidden= [128, 64])
    elif model_name == 'dsmil':
        model = DSMIL(in_dim, out_dim, d_tiles_values= 32,d_tiles_queries= 32,passing_values= False,tiles_scores_mlp_hidden=[200,100],
                        tiles_queries_mlp_hidden=[200,100], mlp_hidden=[200,100])
    elif model_name == 'hiptmil':
        model = HIPTMIL(in_dim, out_dim)
    elif model_name == 'transmil':
        model = TransMIL(in_dim, out_features=out_dim)
    elif model_name == 'meanpool':
        model = MeanPool(in_dim, out_dim)
    else:
        raise 'model not found'
    return model


In [2]:
device1 = 'cuda:0'

device2 = 'cuda:1'
device1 = device2 = 'cuda:0'

In [3]:
print("torch.cuda.memory_allocated: %fGB"%(torch.cuda.memory_allocated(0)/1024/1024/1024))
print("torch.cuda.memory_reserved: %fGB"%(torch.cuda.memory_reserved(0)/1024/1024/1024))
print("torch.cuda.max_memory_reserved: %fGB"%(torch.cuda.max_memory_reserved(0)/1024/1024/1024))

torch.cuda.memory_allocated: 0.000000GB
torch.cuda.memory_reserved: 0.000000GB
torch.cuda.max_memory_reserved: 0.000000GB


In [4]:
class IBOTMultiTaskModel(nn.Module):
    def __init__(self, num_classes):
        super(IBOTMultiTaskModel, self).__init__()
        weights_path = '/home/yuxin/Downloads/ibot_vit_base_pancan.pth'
        self.base_model = iBOTViT(architecture="vit_base_pancan", encoder="teacher", weights_path=weights_path)
        # print(self.base_model)
        self.num_features = 768
        self.num_classes = num_classes

        if isinstance(num_classes, list):
            self.heads = nn.ModuleList([nn.Linear(self.num_features, num_class) for num_class in num_classes])
        else:
            self.head = self.base_model.head
            self.head.fc = nn.Linear(self.num_features, num_classes)

    def forward(self, x):
        # Forward pass through the base model
        x = self.base_model(x)
        if isinstance(self.num_classes, list):
            x = [head(x) for head in self.heads]
        else:
            x = self.head(x)
        return x

In [5]:
# device = "cuda:0" if torch.cuda.is_available() else "cpu"
# device = 'cuda'
num_classes = [3,3]
img_size = patch_size = 384
data_trans = {
    "train": transforms.Compose([
                                transforms.Resize(img_size),
                                transforms.ColorJitter(),
                                transforms.RandomHorizontalFlip(),
                                transforms.RandomVerticalFlip(),
                                transforms.ToTensor(),
                                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    "test": transforms.Compose([
                                transforms.Resize((img_size,img_size)),
                                transforms.ToTensor(),
                                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])}

In [6]:

model = IBOTMultiTaskModel(num_classes)

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,  # ← use FEATURE_EXTRACTION, not IMAGE_CLASSIFICATION
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.2,
    target_modules=["qkv"],
)
model = get_peft_model(model, lora_config)
model = nn.DataParallel(model)
model.to(device1)

ckpt_path = '/mnt/hd1/bcacad/timm_lora_ft/2025_07_14_04_18_54/model-13.pth'
ckpt = torch.load(ckpt_path, map_location=device1)
model.load_state_dict(ckpt['model_state_dict'])
feature_extractor = model.module.base_model.base_model.eval()


2026-02-16 04:44:57.680 | INFO     | rl_benchmarks.models.feature_extractors.ibot_vit:__init__:78 - Pretrained weights found at /home/yuxin/Downloads/ibot_vit_base_pancan.pth and loaded with msg: _IncompatibleKeys(missing_keys=[], unexpected_keys=['head.mlp.0.weight', 'head.mlp.0.bias', 'head.mlp.2.weight', 'head.mlp.2.bias', 'head.mlp.4.weight', 'head.mlp.4.bias', 'head.last_layer.weight_g', 'head.last_layer.weight_v', 'head.last_layer2.weight_g', 'head.last_layer2.weight_v'])


In [7]:
class Feature_Fly(Dataset):
    def get_label(self, type):
        D = {
            'normal':[0,0],
            'dcis-1':[1,0],
            'dcis-2':[1,1],
            'dcis-3':[1,2],
            'ibc-1':[2,0],
            'ibc-2':[2,1],
            'ibc-3':[2,2],
        }
        label =  D[type]
        return label[0],label[1]
    
    def __init__(self, image_dir, phase='test', train_tile=None):
        folder = Path(image_dir)
        self.im_paths = list(folder.rglob('**/*.png'))
        self.labels = [self.get_label(path.parent.name) for path in self.im_paths]
        self.phase=phase
        self.train_tile = train_tile
        self.size = 336
        self.bs = 16
        self.src_mag = 10
        self.tar_mag = 10
    
        
    def __len__(self):
        return len(self.im_paths)
    def __getitem__(self, item):
        im_path = self.im_paths[item]
        im = QiLuROI(str(im_path), self.src_mag, self.tar_mag, self.size)
        im.setIterator(self.size)
        patches = [p for p in im]
        if self.phase == 'train':
            indices = np.random.choice(len(patches), self.train_tile, replace=True)
            patches = [patches[i] for i in indices]
        patches = [data_trans[self.phase](p) for p in patches]
        bs = self.bs
        for i in range(0, len(patches), bs):
            x = torch.stack(patches[i:i+bs], dim=0)
            x = x.to(device1)
            y = feature_extractor(x)
            if i == 0:
                features = y.detach().cpu().numpy()
            else:
                # features = torch.concatenate([features, y], dim=0)
                np.concatenate([features, y.detach().cpu().numpy()], axis=0)
        return features, self.labels[item]
    
    @staticmethod
    def collate_fn(batch):
        # 官方实现的default_collate可以参考
        # https://github.com/pytorch/pytorch/blob/67b7e751e6b5931a9f45274653f4f653a4e6cdf6/torch/utils/data/_utils/collate.py
        images,  labels = tuple(zip(*batch))

        images = torch.stack(images, dim=0)
        labels = torch.as_tensor(labels)
        # masks = torch.as_tensor(masks)

        return images, labels

class Feature(Dataset):
    def get_label(self, type):
        D = {
            'normal':[0,0],
            'dcis-1':[1,0],
            'dcis-2':[1,1],
            'dcis-3':[1,2],
            'ibc-1':[2,0],
            'ibc-2':[2,1],
            'ibc-3':[2,2],
        }
        label =  D[type]
        return label[0],label[1]
    
    def __init__(self, feature_dir, phase='test'):
        folder = Path(feature_dir)
        feature_paths = list(folder.rglob('**/*.npy'))
        self.labels = [self.get_label(path.parent.name) for path in feature_paths]
        self.features = [np.load(path) for path in feature_paths]
        self.phase=phase
        
    def __len__(self):
        return len(self.features)
    def __getitem__(self, item):
        features = self.features[item]
        if self.phase == 'train':
            indices = np.random.choice(features.shape[0], 8, replace=True)
            features = np.stack([features[i] for i in indices], axis=0)
        return features, self.labels[item]
    
    @staticmethod
    def collate_fn(batch):
        # 官方实现的default_collate可以参考
        # https://github.com/pytorch/pytorch/blob/67b7e751e6b5931a9f45274653f4f653a4e6cdf6/torch/utils/data/_utils/collate.py
        images,  labels = tuple(zip(*batch))
        images = torch.asTensor(images).to('cuda:1')
        images = torch.stack(images, dim=0)
        labels = torch.as_tensor(labels)
        # masks = torch.as_tensor(masks)

        return images, labels


In [8]:
cfg = dict(
    epoch=2,
    bs=8,
    train_tiles = 4,
)

In [9]:
data_root = Path('/mnt/hd0/project/bcacad/data/roi-level')

train_set = Feature_Fly(data_root / 'suqh' / 'model', 'train', cfg['train_tiles'])

In [10]:
criterion = nn.CrossEntropyLoss()
merics = {'acc': compute_multiclass_accuracy, 'auc': compute_mean_one_vs_all_auc}
input_dim = 768
out_dim = [3,3]
tasks = ['type', 'nonibc', 'ibc']
class_names = {
    'type': ['Normal', 'nonIBC', 'IBC'],
    'nonibc': ['Low', 'Medium', 'High'],
    'ibc': ['Low', 'Medium', 'High'],
}

In [11]:
# model_names = [ 'abmil',  'transmil']
model_names = ['abmil']

models={}
for model_name in model_names:
    # print(model_name)
    model = get_model(model_name, input_dim, out_dim).to(device1)
    trainer = TorchTrainer(model, criterion, merics, device=device1, num_epochs=cfg['epoch'], batch_size=cfg['bs'])
    res = trainer.train(train_set, train_set)
    # res = trainer.train(fake, fake)
    models[model_name] = model


    pha = 'train'
    labels = np.array(res[pha][0])
    probs = np.array(res[pha][1])
    preds = np.array(res[pha][2])

    type_labels = labels[0]
    type_probs = probs[0]
    type_preds = preds[0]

    nonibc_index = np.where(type_labels ==1)
    nonibc_labels = labels[1][nonibc_index]
    nonibc_probs = probs[1][nonibc_index]
    nonibc_preds = preds[1][nonibc_index]

    ibc_index = np.where(type_labels ==2)
    ibc_labels = labels[1][ibc_index]
    ibc_probs = probs[1][ibc_index]
    ibc_preds = preds[1][ibc_index]

    re = {}
    avg_aucs = {}
    re['type'] = report(type_labels, type_preds, type_probs, class_names['type'])
    avg_aucs['type'] = compute_mean_one_vs_all_auc(type_labels, type_probs)

    re['nonibc'] = report(nonibc_labels, nonibc_preds, nonibc_probs, class_names['nonibc'])
    avg_aucs['nonibc'] = compute_mean_one_vs_all_auc(nonibc_labels, nonibc_probs)

    re['ibc'] = report(ibc_labels, ibc_preds, ibc_probs, class_names['ibc'])
    avg_aucs['ibc'] = compute_mean_one_vs_all_auc(ibc_labels, ibc_probs)

    for task in ['type', 'nonibc', 'ibc']:
        r = re[task]
        fs = "{} {} {} acc: {:.4f}, auc: {:.4f} [{:.4f} {:.4f} {:.4f}]".format(model_name, pha, task, r['accuracy'], avg_aucs[task], r['0']['auc'], r['1']['auc'], r['2']['auc'])
        print(fs)
    print()
print()
    

    


Epoch 0
loss: 0.4302922487258911: 100%|██████████| 626/626 [05:11<00:00,  2.01it/s] 
Epoch 0: train_loss 0.6430389881134033
Epoch 1
loss: 0.1954791396856308: 100%|██████████| 626/626 [05:04<00:00,  2.06it/s] 
Epoch 1: train_loss 0.3788261413574219
abmil train type acc: 0.9491, auc: 0.9871 [0.9986 0.9789 0.9835]
abmil train nonibc acc: 0.6263, auc: 0.7735 [0.8803 0.6060 0.8325]
abmil train ibc acc: 0.7419, auc: 0.8416 [0.9003 0.7578 0.8670]




In [14]:
models[model_name] = model

In [12]:
save_root = Path('/mnt/hd0/project/bcacad/model/roi_models_lora/model3_epoch2')
if not save_root.exists():
    save_root.mkdir(parents=True, exist_ok=True)
for model_name in model_names:
    torch.save(models[model_name].state_dict(), save_root / f'{model_name}.pth')
    print(f'save {model_name}.pth')

save abmil.pth


In [11]:
model_names = [ 'abmil']
models={}
model_dir = Path('/mnt/hd0/project/bcacad/model/roi_models_lora/model3_epoch2')
for model_name in model_names:
    model = get_model(model_name, input_dim, out_dim).to(device1)
    model.load_state_dict(torch.load(model_dir / f'{model_name}.pth'))
    models[model_name] = model.to(device1)

In [12]:
def numpy_softmax(logits: np.ndarray) -> np.ndarray:
    # logits: shape (n_samples, n_classes)
    # subtract max per row for numerical stability
    exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    return exps / np.sum(exps, axis=1, keepdims=True)

In [13]:
import numpy as np
import scipy.stats as st
from sklearn.metrics import confusion_matrix, cohen_kappa_score

def kappa_ci(y_true, y_pred, weights='linear', alpha=0.05):
    """
    Returns (kappa, lower, upper) for Cohen’s κ using the large‐sample SE:
       SE = sqrt( p0*(1−p0) / [ N*(1−pe)^2 ] )
    where p0 is observed agreement and pe the chance agreement.
    """
    # 1) point estimate
    κ = cohen_kappa_score(y_true, y_pred, weights=weights)

    # 2) observed & expected agreement from the confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    N = cm.sum()
    p0 = np.trace(cm) / N
    marg_true = cm.sum(axis=1) / N
    marg_pred = cm.sum(axis=0) / N
    pe = (marg_true * marg_pred).sum()

    # 3) standard error
    se = np.sqrt( p0*(1-p0) / (N * (1-pe)**2) )

    # 4) CI
    z = st.norm.ppf(1 - alpha/2)
    lower = κ - z*se
    upper = κ + z*se
    return κ, lower, upper




In [14]:
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
import scipy.stats as st

def macro_auc_ci(y_true, y_score, N=3, alpha=0.05):
    """
    Computes macro-average AUC and confidence interval for multi-class classification.

    Parameters:
        y_true (array-like): True class labels (shape: [n_samples]).
        y_score (array-like): Predicted scores/probabilities (shape: [n_samples, n_classes]).
        alpha (float): Significance level for the confidence interval (default: 0.05).

    Returns:
        (float, float, float): Tuple of (macro_auc, lower_ci, upper_ci)
    """
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    # Binarize true labels for one-vs-rest AUC calculation
    classes = np.arange(N)
    y_true_bin = label_binarize(y_true, classes=classes)

    # Compute AUC for each class (one-vs-rest)
    aucs = []
    for i in range(len(classes)):
        try:
            auc = roc_auc_score(y_true_bin[:, i], y_score[:, i])
            aucs.append(auc)
        except ValueError:
            # If only one class present in y_true_bin[:, i], skip it
            continue

    A = np.mean(aucs)
    se = np.std(aucs, ddof=1) / np.sqrt(len(aucs))  # Standard error of the mean

    z = st.norm.ppf(1 - alpha / 2)
    lower = A - z * se
    upper = A + z * se

    return A, lower, upper


In [15]:
# import os
# import matplotlib.pyplot as plt
# from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# def save_cm_plot(y_true, y_pred, cohort, model_name, task, label_names, save_dir):
#     """
#     Saves a confusion matrix plot to the specified directory.

#     Parameters:
#         y_true (list or array): True labels.
#         y_pred (list or array): Predicted labels.
#         cohort (str): Name of the cohort (e.g., 'test', 'validation').
#         model_name (str): Identifier for the model.
#         task (str): Task name (e.g., 'classification').
#         label_names (list): List of class label names.
#         save_dir (str): Directory to save the plot.

#     Returns:
#         str: Path to the saved confusion matrix plot.
#     """
#     # Create confusion matrix
#     cm = confusion_matrix(y_true, y_pred, labels = list(range(len(label_names))))
#     disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)

#     # Plot the confusion matrix
#     fig, ax = plt.subplots(figsize=(8, 6))
#     disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
#     plt.title(f'Confusion Matrix - {model_name} ({cohort}, {task})')

#     # Ensure save directory exists
#     os.makedirs(save_dir, exist_ok=True)

#     # Create filename and save
#     filename = f"{model_name}_{cohort}_{task}_confusion_matrix.png"
#     filepath = os.path.join(save_dir, filename)
#     plt.savefig(filepath, bbox_inches='tight')
#     plt.close(fig)

#     return filepath
import os
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

def save_cm_plot(y_true, y_pred, cohort, model_name, task, label_names, save_dir, cell_fontsize=28):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(label_names))))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)

    fig, ax = plt.subplots(figsize=(8, 6))
    disp.plot(
        ax=ax,
        cmap="Blues",
        xticks_rotation=45,
        text_kw={"fontsize": cell_fontsize, "fontweight": "bold"}  # <-- cell text
    )
    ax.set_title(f"Confusion Matrix - {model_name} ({cohort}, {task})")

    os.makedirs(save_dir, exist_ok=True)
    filepath = os.path.join(save_dir, f"{model_name}_{cohort}_{task}_confusion_matrix.png")
    plt.savefig(filepath, bbox_inches="tight", dpi=300)
    plt.close(fig)
    return filepath


In [25]:
# testing
import roc_utils as ru
params = {
    # 'font.family':'serif',
    # 'font.serif':'Times New Roman',
    # 'font.style':'normal',
    # 'font.weight':'normal',
    'legend.fontsize': 8,
    'legend.frameon': False,  # remove legend border
    'figure.figsize': (7, 5),
}
bbox = (0.4, 0.38)
from matplotlib import rcParams
rcParams.update(params)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from pathlib import Path

def plot_multiclass_roc(y_true_dict, y_prob_dict, task, save_dir, cohort, model_name, label_names):
    """
    Plot ROC curves with bootstrap confidence intervals for binary or multi-class classification.

    Args:
        y_true_dict (dict): Dictionary mapping task names to ground truth arrays.
        y_prob_dict (dict): Dictionary mapping task names to probability arrays.
        task (str): Task name to access y_true and y_prob.
        save_dir (Path or str): Directory to save the plot.
        cohort (str): Name of the cohort.
        model_name (str): Name of the model.
        label_names (list or dict): Class names to label ROC curves.

    Returns:
        dict: Mean ROC AUC for each class (or for binary task).
    """

    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    y_true = np.array(y_true_dict[task])
    y_prob = np.array(y_prob_dict[task])


    n_classes = len(label_names)
    y_true_bin = label_binarize(y_true, classes=np.arange(n_classes))
    for i in range(n_classes):
        ax = plt.gca()
        class_label = label_names[i]
        ru.plot_roc_bootstrap(X=y_prob[:, i],
                            y=y_true_bin[:, i],
                            pos_label=True,
                            ax=ax,
                            n_bootstrap=1000,
                            title="",
                            label='',
                            show_details=True
                            )

        # Finalize plot
        
        ax.legend()
        # plt.title(title)
        save_path = save_dir / f'{cohort}_{task}_{class_label}_roc.png'
        plt.savefig(save_path, dpi=600)
        plt.close()



In [26]:
pha='test'
feature_root = Path('/mnt/hd0/project/bcacad/model/roi_features_ft_lora')
save_dir = Path('/mnt/hd0/project/bcacad/model/roi_models_lora/model3_epoch2/cm_v2')

cohort_tasks = {
        'suqh_full': ['type', 'nonibc', 'ibc'],
        'qduh': ['type','nonibc', 'ibc'],
        'shsu': ['type','nonibc', 'ibc'],
        'bracs': ['type'],
        'bcnb': ['ibc'],
        'bach': ['type'],
        'apght': ['ibc'],
        'aggregate': ['type', 'nonibc', 'ibc']
}
model = models[model_name]

label_names = {
    'type':['Normal', 'nonIBC', 'IBC'],
    'nonibc': ['Low', 'Medium', 'High'],
    'ibc': ['Low', 'Medium', 'High'],
}
log_file = save_dir / f'evaluation_{time.strftime("%Y_%m_%d_%H_%M_%S")}.txt'
with open(log_file, 'w') as f:
    for cohort, tasks in cohort_tasks.items():
        y_true = {task: [] for task in tasks}
        y_pred = {task: [] for task in tasks}
        y_prob = {task: [] for task in tasks}
        test_set = Feature(feature_root / cohort / 'test', 'test')
        trainer = TorchTrainer(model, criterion, merics, device=device1)
        res = trainer.predict(test_set)

        labels = res[0]
        labels = torch.tensor(labels)
        outputs = res[1]
        outputs = torch.tensor(outputs)
        # print(outputs[0].__class__, outputs[1].shape)

        if 'type' in tasks:
            type_output = outputs[0]
            type_pred = torch.max(type_output, dim=1)[1]
            type_label = labels[0,:]
            y_pred['type'].extend(type_pred.cpu().tolist())
            y_true['type'].extend(type_label.cpu().tolist())
            y_prob['type'].extend(numpy_softmax(type_output.cpu().numpy()))
            
                # Handle grade classification for nonIBC
        if 'nonibc' in tasks:
            nonibc_mask = labels[0,:] == 1  # nonIBC cases
            if nonibc_mask.any():
                grade_output = outputs[1][nonibc_mask]
                grade_pred = torch.max(grade_output, dim=1)[1]
                grade_label = labels[1,nonibc_mask]
                y_pred['nonibc'].extend(grade_pred.cpu().tolist())
                y_true['nonibc'].extend(grade_label.cpu().tolist())
                y_prob['nonibc'].extend(numpy_softmax(grade_output.cpu().numpy()))
            
        # Handle grade classification for IBC
        if 'ibc' in tasks:
            ibc_mask = labels[0,:] == 2  # IBC cases
            if ibc_mask.any():
                grade_output = outputs[1][ibc_mask]
                grade_pred = torch.max(grade_output, dim=1)[1]
                grade_label = labels[1,ibc_mask]
                y_pred['ibc'].extend(grade_pred.cpu().tolist())
                y_true['ibc'].extend(grade_label.cpu().tolist())
                y_prob['ibc'].extend(numpy_softmax(grade_output.cpu().numpy()))
        
        metrics = {}
        
        for task in tasks:
            if len(y_true[task]) > 0:  # Only calculate metrics if we have predictions
                avg_auc, avg_auc_lower, avg_auc_upper = macro_auc_ci(np.array(y_true[task]), np.array(y_prob[task]))
                kappa, kappa_lower, kappa_upper = kappa_ci(np.array(y_true[task]), np.array(y_pred[task]), weights='linear')
                metrics[task] = {
                    'balanced_acc': round(balanced_accuracy_score(y_true[task], y_pred[task]), 4),
                    'f1_macro': round(f1_score(y_true[task], y_pred[task], average='macro'), 4),
                    # 'kappa': round(cohen_kappa_score(y_true[task], y_pred[task], weights='linear'), 4),
                    # 'macro_auroc': round(roc_auc_score(y_true[task], y_prob[task], average='macro', multi_class='ovr'), 4),
                    # 'avg_auc': round(compute_mean_one_vs_all_auc(np.array(y_true[task]), np.array(y_prob[task])), 4),
                    'macro_auroc': round(avg_auc, 4),
                    'macro_auroc_lower': round(avg_auc_lower, 4),
                    'macro_auroc_upper': round(avg_auc_upper, 4),
                    'kappa': round(kappa, 4),
                    'kappa_lower': round(kappa_lower, 4),
                    'kappa_upper': round(kappa_upper, 4),
                    
                }
            else:
                metrics[task] = {
                    'balanced_acc': None,
                    'kappa': None,
                    'f1_macro': None
                }
            # draw confusion matrix
            save_cm_plot(y_true[task], y_pred[task], cohort, model_name, task, label_names[task], save_dir)

            # draw roc curve
            if cohort == 'aggregate':
                plot_multiclass_roc(y_true_dict=y_true, y_prob_dict=y_prob, task=task, cohort = cohort, model_name=
                                    model_name, save_dir=save_dir, label_names=label_names[task])
        
        f.write(f"\n{cohort.upper()}:")
        print(f"\n{cohort.upper()}:", end="")
        for task, task_metrics in metrics.items():
            f.write(f"\n{task.upper()}:")
            print(f"\n{task.upper()}:", end="")
            for metric_name, value in task_metrics.items():
                if value is not None:
                    f.write(f"\t{metric_name}: {value:.4f}")
                    print(f"\t{metric_name}: {value:.4f}", end="")
                else:
                    print(f"\t{metric_name}: N/A", end="")
        print()
        f.write("\n")



AGGREGATE:
TYPE:	balanced_acc: 0.9245	f1_macro: 0.8698	macro_auroc: 0.9815	macro_auroc_lower: 0.9699	macro_auroc_upper: 0.9931	kappa: 0.8421	kappa_lower: 0.8326	kappa_upper: 0.8517
NONIBC:	balanced_acc: 0.5786	f1_macro: 0.5116	macro_auroc: 0.7517	macro_auroc_lower: 0.6856	macro_auroc_upper: 0.8178	kappa: 0.3184	kappa_lower: 0.2852	kappa_upper: 0.3515
IBC:	balanced_acc: 0.4273	f1_macro: 0.3984	macro_auroc: 0.6404	macro_auroc_lower: 0.5216	macro_auroc_upper: 0.7593	kappa: 0.1637	kappa_lower: 0.1463	kappa_upper: 0.1812
